# N-Body Problem: From Simulation to Discovery

**Pipeline:**
1. Simulate → Generate orbital dynamics
2. Visualize → Animate motion
3. Measure → Extract data
4. Discover → Learn governing equations with SINDy & PySR

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp
from IPython.display import HTML
import pysindy as ps
from pysr import PySRRegressor

---
# Part 1: Simulation Setup

## 1.0 Physics Engine

In [ ]:
#==============================================================================
# PHYSICS PARAMETERS
#==============================================================================
G = 1.0  # Gravitational constant (normalized units)

#==============================================================================
# SIMULATION FUNCTIONS
#==============================================================================
def compute_accelerations(positions, masses):
    """
    Compute gravitational accelerations for all bodies.
    
    Parameters:
        positions: (n_bodies, 3) array of positions
        masses: (n_bodies,) array of masses
    
    Returns:
        (n_bodies, 3) array of accelerations
    """
    n = len(masses)
    acc = np.zeros_like(positions)
    for i in range(n):
        for j in range(n):
            if i != j:
                r_ij = positions[j] - positions[i]
                dist = np.linalg.norm(r_ij)
                acc[i] += G * masses[j] * r_ij / dist**3
    return acc

def nbody_derivative(t, state, masses):
    """
    Compute derivatives for n-body ODE.
    
    State vector: [x1,y1,z1, x2,y2,z2, ..., vx1,vy1,vz1, vx2,vy2,vz2, ...]
    """
    n = len(masses)
    positions = state[:3*n].reshape(n, 3)
    velocities = state[3*n:].reshape(n, 3)
    accelerations = compute_accelerations(positions, masses)
    return np.concatenate([velocities.flatten(), accelerations.flatten()])

def run_simulation(masses, positions, velocities, t_span, n_samples, rtol=1e-10):
    """
    Run n-body simulation.
    
    Parameters:
        masses: (n_bodies,) array
        positions: (n_bodies, 3) initial positions
        velocities: (n_bodies, 3) initial velocities
        t_span: (t_start, t_end) tuple
        n_samples: number of time points
        rtol: relative tolerance for ODE solver
    
    Returns:
        t: time array
        positions_all: (n_samples, n_bodies, 3) positions
        velocities_all: (n_samples, n_bodies, 3) velocities
    """
    n = len(masses)
    state0 = np.concatenate([positions.flatten(), velocities.flatten()])
    t_eval = np.linspace(t_span[0], t_span[1], n_samples)
    
    sol = solve_ivp(
        lambda t, s: nbody_derivative(t, s, masses),
        t_span, state0, t_eval=t_eval, method='DOP853', rtol=rtol
    )
    
    if not sol.success:
        raise RuntimeError(f"Simulation failed: {sol.message}")
    
    # Reshape outputs
    pos_all = sol.y[:3*n, :].T.reshape(-1, n, 3)
    vel_all = sol.y[3*n:, :].T.reshape(-1, n, 3)
    
    return sol.t, pos_all, vel_all

print("Physics engine loaded.")

## 1.1 Two-Body Simulation

In [ ]:
#==============================================================================
# TWO-BODY PARAMETERS (EDIT HERE)
#==============================================================================

# Masses
M_sun = 1.0        # Central body mass
M_earth = 0.001    # Orbiting body mass

# Initial position of orbiting body (central body at origin)
r0 = 1.0           # Initial distance from center
theta0 = 0.0       # Initial angle (radians)

# Orbit type: 'circular', 'elliptical', or 'custom'
orbit_type = 'elliptical'
eccentricity = 0.3  # Only used if orbit_type='elliptical' (0=circular, <1=ellipse)

# Simulation settings
t_end_2body = 30.0       # End time
n_samples_2body = 2000   # Number of samples

#==============================================================================
# COMPUTE INITIAL CONDITIONS
#==============================================================================

masses_2body = np.array([M_sun, M_earth])

# Initial position
x0 = r0 * np.cos(theta0)
y0 = r0 * np.sin(theta0)
pos_2body_init = np.array([
    [0.0, 0.0, 0.0],      # Sun at origin
    [x0, y0, 0.0],        # Earth
])

# Initial velocity (perpendicular to position)
v_circular = np.sqrt(G * M_sun / r0)  # Circular orbit speed

if orbit_type == 'circular':
    v0 = v_circular
elif orbit_type == 'elliptical':
    # At perihelion: v = v_circular * sqrt((1+e)/(1-e)) for starting at perihelion
    # At aphelion: v = v_circular * sqrt((1-e)/(1+e))
    v0 = v_circular * np.sqrt((1 + eccentricity))  # Starting at perihelion
else:
    v0 = v_circular  # Custom: modify as needed

# Velocity perpendicular to initial position
vx0 = -v0 * np.sin(theta0)
vy0 = v0 * np.cos(theta0)
vel_2body_init = np.array([
    [0.0, 0.0, 0.0],      # Sun stationary
    [vx0, vy0, 0.0],      # Earth
])

print("Two-Body Initial Conditions:")
print(f"  Sun mass: {M_sun}")
print(f"  Earth mass: {M_earth}")
print(f"  Initial distance: {r0}")
print(f"  Orbit type: {orbit_type}")
print(f"  Initial speed: {v0:.4f} (circular would be {v_circular:.4f})")
print(f"  Samples: {n_samples_2body}")

In [ ]:
#==============================================================================
# RUN TWO-BODY SIMULATION
#==============================================================================

t_2body, pos_2body, vel_2body = run_simulation(
    masses_2body,
    pos_2body_init,
    vel_2body_init,
    t_span=(0, t_end_2body),
    n_samples=n_samples_2body
)

# Extract individual bodies
pos_sun = pos_2body[:, 0, :]    # (n_samples, 3)
pos_earth = pos_2body[:, 1, :]
vel_sun = vel_2body[:, 0, :]
vel_earth = vel_2body[:, 1, :]

print(f"\nSimulation complete!")
print(f"  Time points: {len(t_2body)}")
print(f"  Time step (dt): {t_2body[1] - t_2body[0]:.6f}")
print(f"  Position shape: {pos_earth.shape}")

## 1.2 Three-Body Simulation

In [ ]:
#==============================================================================
# THREE-BODY PARAMETERS (EDIT HERE)
#==============================================================================

# Masses (equal masses create interesting chaotic dynamics)
M1 = 1.0
M2 = 1.0
M3 = 1.0

# Initial configuration: 'triangle', 'line', 'figure8', or 'custom'
config_type = 'triangle'

# Triangle configuration parameters
triangle_size = 1.0      # Side length of triangle
velocity_scale = 0.4     # Scale of initial velocities

# Simulation settings
t_end_3body = 40.0       # End time (longer for chaotic dynamics)
n_samples_3body = 4000   # More samples for complex motion

#==============================================================================
# COMPUTE INITIAL CONDITIONS
#==============================================================================

masses_3body = np.array([M1, M2, M3])

if config_type == 'triangle':
    # Equilateral triangle
    pos_3body_init = triangle_size * np.array([
        [-0.5, 0.0, 0.0],
        [0.5, 0.0, 0.0],
        [0.0, np.sqrt(3)/2, 0.0],
    ])
    vel_3body_init = velocity_scale * np.array([
        [0.0, 1.0, 0.0],
        [0.0, -1.0, 0.0],
        [1.0, 0.0, 0.0],
    ])

elif config_type == 'line':
    # Bodies in a line
    pos_3body_init = np.array([
        [-1.0, 0.0, 0.0],
        [0.0, 0.0, 0.0],
        [1.0, 0.0, 0.0],
    ])
    vel_3body_init = velocity_scale * np.array([
        [0.0, 0.5, 0.0],
        [0.0, -1.0, 0.0],
        [0.0, 0.5, 0.0],
    ])

elif config_type == 'figure8':
    # Famous figure-8 periodic solution (Chenciner & Montgomery)
    pos_3body_init = np.array([
        [-0.97000436, 0.24308753, 0.0],
        [0.97000436, -0.24308753, 0.0],
        [0.0, 0.0, 0.0],
    ])
    v = 0.4662036850
    vel_3body_init = np.array([
        [v, v, 0.0],
        [v, v, 0.0],
        [-2*v, -2*v, 0.0],
    ])

else:  # custom
    pos_3body_init = np.array([
        [-0.5, 0.0, 0.0],
        [0.5, 0.0, 0.0],
        [0.0, 1.0, 0.0],
    ])
    vel_3body_init = np.array([
        [0.0, 0.3, 0.0],
        [0.0, -0.3, 0.0],
        [0.3, 0.0, 0.0],
    ])

# Ensure zero total momentum (center of mass stationary)
total_momentum = np.sum(masses_3body[:, None] * vel_3body_init, axis=0)
vel_3body_init -= total_momentum / np.sum(masses_3body)

print("Three-Body Initial Conditions:")
print(f"  Masses: {masses_3body}")
print(f"  Configuration: {config_type}")
print(f"  Samples: {n_samples_3body}")
print(f"  Total momentum: {np.sum(masses_3body[:, None] * vel_3body_init, axis=0)} (should be ~0)")

In [ ]:
#==============================================================================
# RUN THREE-BODY SIMULATION
#==============================================================================

t_3body, pos_3body, vel_3body = run_simulation(
    masses_3body,
    pos_3body_init,
    vel_3body_init,
    t_span=(0, t_end_3body),
    n_samples=n_samples_3body
)

# Extract individual bodies
pos_b1 = pos_3body[:, 0, :]
pos_b2 = pos_3body[:, 1, :]
pos_b3 = pos_3body[:, 2, :]
vel_b1 = vel_3body[:, 0, :]
vel_b2 = vel_3body[:, 1, :]
vel_b3 = vel_3body[:, 2, :]

print(f"\nSimulation complete!")
print(f"  Time points: {len(t_3body)}")
print(f"  Time step (dt): {t_3body[1] - t_3body[0]:.6f}")

---
# Part 2: Visualization

In [ ]:
#==============================================================================
# STATIC PLOTS
#==============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Two-body orbit
ax1.plot(pos_sun[:, 0], pos_sun[:, 1], 'yo', markersize=12, label='Sun', zorder=5)
ax1.plot(pos_earth[:, 0], pos_earth[:, 1], 'b-', linewidth=0.5, alpha=0.7, label='Earth orbit')
ax1.plot(pos_earth[0, 0], pos_earth[0, 1], 'bo', markersize=8, label='Earth start')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_aspect('equal')
ax1.set_title(f'Two-Body ({orbit_type} orbit)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Three-body orbits
ax2.plot(pos_b1[:, 0], pos_b1[:, 1], 'r-', linewidth=0.5, alpha=0.7, label='Body 1')
ax2.plot(pos_b2[:, 0], pos_b2[:, 1], 'g-', linewidth=0.5, alpha=0.7, label='Body 2')
ax2.plot(pos_b3[:, 0], pos_b3[:, 1], 'b-', linewidth=0.5, alpha=0.7, label='Body 3')
ax2.plot(pos_b1[0, 0], pos_b1[0, 1], 'ro', markersize=10)
ax2.plot(pos_b2[0, 0], pos_b2[0, 1], 'go', markersize=10)
ax2.plot(pos_b3[0, 0], pos_b3[0, 1], 'bo', markersize=10)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_aspect('equal')
ax2.set_title(f'Three-Body ({config_type} config)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#==============================================================================
# ANIMATION PARAMETERS (EDIT HERE)
#==============================================================================

frame_skip = 5        # Skip frames to speed up animation
interval_ms = 30      # Milliseconds between frames
trail_length = 200    # Number of points in trail (None = full trail)

#==============================================================================
# ANIMATION
#==============================================================================

# Downsample for animation
idx = np.arange(0, min(len(t_2body), len(t_3body)), frame_skip)
n_frames = len(idx)

# Axis limits
margin = 0.3
xlim_2 = (pos_earth[:, 0].min() - margin, pos_earth[:, 0].max() + margin)
ylim_2 = (pos_earth[:, 1].min() - margin, pos_earth[:, 1].max() + margin)

all_x_3 = np.concatenate([pos_b1[:, 0], pos_b2[:, 0], pos_b3[:, 0]])
all_y_3 = np.concatenate([pos_b1[:, 1], pos_b2[:, 1], pos_b3[:, 1]])
xlim_3 = (all_x_3.min() - margin, all_x_3.max() + margin)
ylim_3 = (all_y_3.min() - margin, all_y_3.max() + margin)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.set_xlim(xlim_2)
ax1.set_ylim(ylim_2)
ax1.set_aspect('equal')
ax1.set_title('Two-Body')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.grid(True, alpha=0.3)

ax2.set_xlim(xlim_3)
ax2.set_ylim(ylim_3)
ax2.set_aspect('equal')
ax2.set_title('Three-Body')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.grid(True, alpha=0.3)

# Plot objects
sun, = ax1.plot([], [], 'yo', markersize=15)
earth, = ax1.plot([], [], 'bo', markersize=8)
trail_e, = ax1.plot([], [], 'b-', linewidth=0.5, alpha=0.5)
time_text1 = ax1.text(0.02, 0.98, '', transform=ax1.transAxes, va='top')

b1, = ax2.plot([], [], 'ro', markersize=10)
b2, = ax2.plot([], [], 'go', markersize=10)
b3, = ax2.plot([], [], 'bo', markersize=10)
tr1, = ax2.plot([], [], 'r-', linewidth=0.5, alpha=0.4)
tr2, = ax2.plot([], [], 'g-', linewidth=0.5, alpha=0.4)
tr3, = ax2.plot([], [], 'b-', linewidth=0.5, alpha=0.4)
time_text2 = ax2.text(0.02, 0.98, '', transform=ax2.transAxes, va='top')

def init():
    for obj in [sun, earth, trail_e, b1, b2, b3, tr1, tr2, tr3]:
        obj.set_data([], [])
    time_text1.set_text('')
    time_text2.set_text('')
    return sun, earth, trail_e, b1, b2, b3, tr1, tr2, tr3, time_text1, time_text2

def animate(frame):
    i = idx[frame]
    
    # Trail start index
    trail_start = max(0, i - trail_length) if trail_length else 0
    
    # Two-body
    sun.set_data([pos_sun[i, 0]], [pos_sun[i, 1]])
    earth.set_data([pos_earth[i, 0]], [pos_earth[i, 1]])
    trail_e.set_data(pos_earth[trail_start:i+1, 0], pos_earth[trail_start:i+1, 1])
    time_text1.set_text(f't = {t_2body[i]:.2f}')
    
    # Three-body
    b1.set_data([pos_b1[i, 0]], [pos_b1[i, 1]])
    b2.set_data([pos_b2[i, 0]], [pos_b2[i, 1]])
    b3.set_data([pos_b3[i, 0]], [pos_b3[i, 1]])
    tr1.set_data(pos_b1[trail_start:i+1, 0], pos_b1[trail_start:i+1, 1])
    tr2.set_data(pos_b2[trail_start:i+1, 0], pos_b2[trail_start:i+1, 1])
    tr3.set_data(pos_b3[trail_start:i+1, 0], pos_b3[trail_start:i+1, 1])
    time_text2.set_text(f't = {t_3body[i]:.2f}')
    
    return sun, earth, trail_e, b1, b2, b3, tr1, tr2, tr3, time_text1, time_text2

anim = FuncAnimation(fig, animate, init_func=init, frames=n_frames, interval=interval_ms, blit=True)
plt.close()
HTML(anim.to_jshtml())

---
# Part 3: Data Extraction

Extract measurements for learning algorithms.

## 3.1 Two-Body Measurements

In [ ]:
#==============================================================================
# COMPUTE DERIVED QUANTITIES
#==============================================================================

dt_2body = t_2body[1] - t_2body[0]

# Relative position and distance
r_vec_2body = pos_earth - pos_sun
distance_2body = np.linalg.norm(r_vec_2body, axis=1)

# Speed
speed_2body = np.linalg.norm(vel_earth, axis=1)

# Acceleration (numerical differentiation)
acc_2body = np.gradient(vel_earth, dt_2body, axis=0)
acc_mag_2body = np.linalg.norm(acc_2body, axis=1)

# Radial and tangential components
r_hat = r_vec_2body / distance_2body[:, None]
acc_radial = np.sum(acc_2body * r_hat, axis=1)
acc_tangent = np.sqrt(acc_mag_2body**2 - acc_radial**2)

# Energy
KE_2body = 0.5 * M_earth * speed_2body**2
PE_2body = -G * M_sun * M_earth / distance_2body
E_total_2body = KE_2body + PE_2body

# Angular momentum (z-component for 2D motion)
L_2body = M_earth * (pos_earth[:, 0] * vel_earth[:, 1] - pos_earth[:, 1] * vel_earth[:, 0])

print("Two-Body Measurements:")
print(f"  Distance range: [{distance_2body.min():.4f}, {distance_2body.max():.4f}]")
print(f"  Speed range: [{speed_2body.min():.4f}, {speed_2body.max():.4f}]")
print(f"  Energy conservation: {(E_total_2body.max() - E_total_2body.min()) / np.abs(E_total_2body.mean()) * 100:.6f}%")
print(f"  Angular momentum conservation: {(L_2body.max() - L_2body.min()) / np.abs(L_2body.mean()) * 100:.6f}%")

In [ ]:
#==============================================================================
# PLOT TWO-BODY MEASUREMENTS
#==============================================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Distance vs time
axes[0, 0].plot(t_2body, distance_2body)
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Distance r')
axes[0, 0].set_title('Distance')
axes[0, 0].grid(True, alpha=0.3)

# Speed vs time
axes[0, 1].plot(t_2body, speed_2body)
axes[0, 1].set_xlabel('Time')
axes[0, 1].set_ylabel('Speed |v|')
axes[0, 1].set_title('Speed')
axes[0, 1].grid(True, alpha=0.3)

# Acceleration vs time
axes[0, 2].plot(t_2body, acc_mag_2body, label='|a|')
axes[0, 2].plot(t_2body, -acc_radial, '--', label='-a_radial')
axes[0, 2].set_xlabel('Time')
axes[0, 2].set_ylabel('Acceleration')
axes[0, 2].set_title('Acceleration')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# KEY: Acceleration vs 1/r²
axes[1, 0].scatter(1/distance_2body**2, acc_mag_2body, s=2, alpha=0.3)
# Fit line
r_inv_sq = 1/distance_2body**2
fit_coef = np.polyfit(r_inv_sq, acc_mag_2body, 1)
axes[1, 0].plot([r_inv_sq.min(), r_inv_sq.max()], 
                [fit_coef[0]*r_inv_sq.min() + fit_coef[1], fit_coef[0]*r_inv_sq.max() + fit_coef[1]], 
                'r-', linewidth=2, label=f'Fit: a = {fit_coef[0]:.3f}/r²')
axes[1, 0].set_xlabel('1/r²')
axes[1, 0].set_ylabel('Acceleration |a|')
axes[1, 0].set_title('Newton\'s Law: a ∝ 1/r²')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Energy
axes[1, 1].plot(t_2body, KE_2body, label='Kinetic')
axes[1, 1].plot(t_2body, PE_2body, label='Potential')
axes[1, 1].plot(t_2body, E_total_2body, 'k--', label='Total')
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('Energy')
axes[1, 1].set_title('Energy Conservation')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Angular momentum
axes[1, 2].plot(t_2body, L_2body)
axes[1, 2].set_xlabel('Time')
axes[1, 2].set_ylabel('Angular Momentum L')
axes[1, 2].set_title('Angular Momentum Conservation')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3.2 Three-Body Measurements

In [ ]:
#==============================================================================
# COMPUTE DERIVED QUANTITIES
#==============================================================================

dt_3body = t_3body[1] - t_3body[0]

# Pairwise distances
dist_12 = np.linalg.norm(pos_b2 - pos_b1, axis=1)
dist_13 = np.linalg.norm(pos_b3 - pos_b1, axis=1)
dist_23 = np.linalg.norm(pos_b3 - pos_b2, axis=1)

# Speeds
speed_b1 = np.linalg.norm(vel_b1, axis=1)
speed_b2 = np.linalg.norm(vel_b2, axis=1)
speed_b3 = np.linalg.norm(vel_b3, axis=1)

# Accelerations
acc_b1 = np.gradient(vel_b1, dt_3body, axis=0)
acc_b2 = np.gradient(vel_b2, dt_3body, axis=0)
acc_b3 = np.gradient(vel_b3, dt_3body, axis=0)

# Energy
KE_3body = 0.5 * (M1*speed_b1**2 + M2*speed_b2**2 + M3*speed_b3**2)
PE_3body = -G * (M1*M2/dist_12 + M1*M3/dist_13 + M2*M3/dist_23)
E_total_3body = KE_3body + PE_3body

# Angular momentum (z-component)
Lz_b1 = M1 * (pos_b1[:, 0] * vel_b1[:, 1] - pos_b1[:, 1] * vel_b1[:, 0])
Lz_b2 = M2 * (pos_b2[:, 0] * vel_b2[:, 1] - pos_b2[:, 1] * vel_b2[:, 0])
Lz_b3 = M3 * (pos_b3[:, 0] * vel_b3[:, 1] - pos_b3[:, 1] * vel_b3[:, 0])
L_total_3body = Lz_b1 + Lz_b2 + Lz_b3

print("Three-Body Measurements:")
print(f"  Energy conservation: {(E_total_3body.max() - E_total_3body.min()) / np.abs(E_total_3body.mean()) * 100:.4f}%")
print(f"  Angular momentum conservation: {(L_total_3body.max() - L_total_3body.min()) / (np.abs(L_total_3body.mean()) + 1e-10) * 100:.4f}%")

In [ ]:
#==============================================================================
# PLOT THREE-BODY MEASUREMENTS
#==============================================================================

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Pairwise distances
axes[0, 0].plot(t_3body, dist_12, label='d(1,2)')
axes[0, 0].plot(t_3body, dist_13, label='d(1,3)')
axes[0, 0].plot(t_3body, dist_23, label='d(2,3)')
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Distance')
axes[0, 0].set_title('Pairwise Distances')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Speeds
axes[0, 1].plot(t_3body, speed_b1, label='Body 1')
axes[0, 1].plot(t_3body, speed_b2, label='Body 2')
axes[0, 1].plot(t_3body, speed_b3, label='Body 3')
axes[0, 1].set_xlabel('Time')
axes[0, 1].set_ylabel('Speed')
axes[0, 1].set_title('Speeds')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Energy
axes[1, 0].plot(t_3body, KE_3body, label='Kinetic')
axes[1, 0].plot(t_3body, PE_3body, label='Potential')
axes[1, 0].plot(t_3body, E_total_3body, 'k--', linewidth=2, label='Total')
axes[1, 0].set_xlabel('Time')
axes[1, 0].set_ylabel('Energy')
axes[1, 0].set_title('Energy Conservation')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Angular momentum
axes[1, 1].plot(t_3body, L_total_3body)
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('Total Angular Momentum')
axes[1, 1].set_title('Angular Momentum Conservation')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# Part 4: Learning - Discover Governing Equations

## 4.1 PySR: Discover Force Law from Two-Body Data

**Goal**: From (distance, acceleration) → discover $a = GM/r^2$

In [ ]:
#==============================================================================
# PYSR PARAMETERS (EDIT HERE)
#==============================================================================

pysr_niterations = 50          # Number of iterations
pysr_populations = 20          # Number of populations
pysr_population_size = 40      # Size of each population
pysr_maxsize = 15              # Maximum equation complexity

# Operators to include
pysr_binary_ops = ["+", "-", "*", "/"]
pysr_unary_ops = ["square", "inv", "sqrt"]

# Subsample data for faster training
pysr_n_samples = 500  # Number of samples to use

#==============================================================================
# PREPARE DATA
#==============================================================================

# Subsample uniformly
idx_pysr = np.linspace(0, len(distance_2body)-1, pysr_n_samples, dtype=int)

X_pysr = distance_2body[idx_pysr].reshape(-1, 1)
y_pysr = acc_mag_2body[idx_pysr]

print(f"PySR Training Data:")
print(f"  Samples: {len(X_pysr)}")
print(f"  Input (r): shape {X_pysr.shape}")
print(f"  Output (a): shape {y_pysr.shape}")

In [ ]:
#==============================================================================
# FIT PYSR MODEL
#==============================================================================

model_pysr = PySRRegressor(
    niterations=pysr_niterations,
    binary_operators=pysr_binary_ops,
    unary_operators=pysr_unary_ops,
    populations=pysr_populations,
    population_size=pysr_population_size,
    maxsize=pysr_maxsize,
    verbosity=1,
)

model_pysr.fit(X_pysr, y_pysr)

In [ ]:
#==============================================================================
# PYSR RESULTS
#==============================================================================

print("\n" + "="*60)
print("PySR DISCOVERED EQUATIONS (Pareto Front)")
print("="*60)
print(model_pysr)
print("\n" + "-"*60)
print(f"Best equation: a = {model_pysr.sympy()}")
print(f"Expected:      a = {G * M_sun}/r² = 1.0/r²")
print("-"*60)

In [ ]:
#==============================================================================
# VALIDATE PYSR RESULT
#==============================================================================

r_test = np.linspace(distance_2body.min(), distance_2body.max(), 200).reshape(-1, 1)
a_true = G * M_sun / r_test.flatten()**2
a_pred = model_pysr.predict(r_test)

plt.figure(figsize=(10, 6))
plt.scatter(X_pysr, y_pysr, s=10, alpha=0.3, label='Training data')
plt.plot(r_test, a_true, 'r-', linewidth=2, label=f'True: GM/r² = {G*M_sun}/r²')
plt.plot(r_test, a_pred, 'g--', linewidth=2, label=f'PySR: {model_pysr.sympy()}')
plt.xlabel('Distance r', fontsize=12)
plt.ylabel('Acceleration a', fontsize=12)
plt.title('PySR: Discovering the Inverse Square Law', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

# Error metrics
mse = np.mean((a_pred.flatten() - a_true)**2)
print(f"\nMean Squared Error: {mse:.6e}")

## 4.2 SINDy: Discover Equations of Motion

**Goal**: From state trajectory → discover $\dot{x} = v$, $\dot{v} = f(x)$

In [ ]:
#==============================================================================
# SINDY PARAMETERS (EDIT HERE)
#==============================================================================

sindy_threshold = 0.05         # Sparsity threshold (smaller = more terms)
sindy_poly_degree = 2          # Polynomial degree
sindy_include_bias = True      # Include constant term

# Subsample for cleaner derivatives
sindy_n_samples = 1000

#==============================================================================
# PREPARE DATA
#==============================================================================

# Subsample
idx_sindy = np.linspace(0, len(t_2body)-1, sindy_n_samples, dtype=int)

# State: [x, y, vx, vy]
X_sindy = np.column_stack([
    pos_earth[idx_sindy, 0],
    pos_earth[idx_sindy, 1],
    vel_earth[idx_sindy, 0],
    vel_earth[idx_sindy, 1],
])

t_sindy = t_2body[idx_sindy]

print(f"SINDy Training Data:")
print(f"  Samples: {len(X_sindy)}")
print(f"  State shape: {X_sindy.shape}")
print(f"  Variables: [x, y, vx, vy]")

In [ ]:
#==============================================================================
# FIT SINDY MODEL
#==============================================================================

model_sindy = ps.SINDy(
    optimizer=ps.STLSQ(threshold=sindy_threshold),
    feature_library=ps.PolynomialLibrary(degree=sindy_poly_degree, include_bias=sindy_include_bias),
)

model_sindy.fit(X_sindy, t=t_sindy, feature_names=['x', 'y', 'vx', 'vy'])

print("\n" + "="*60)
print("SINDy DISCOVERED EQUATIONS")
print("="*60)
model_sindy.print()

In [ ]:
#==============================================================================
# VALIDATE SINDY MODEL
#==============================================================================

# Simulate using discovered equations
x0_sindy = X_sindy[0]
X_sim = model_sindy.simulate(x0_sindy, t_sindy)

# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

labels = ['x', 'y', 'vx', 'vy']
for i, ax in enumerate(axes.flat):
    ax.plot(t_sindy, X_sindy[:, i], 'k-', linewidth=1, label='True')
    ax.plot(t_sindy, X_sim[:, i], 'r--', linewidth=1, label='SINDy')
    ax.set_xlabel('Time')
    ax.set_ylabel(labels[i])
    ax.set_title(f'{labels[i]} vs Time')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('SINDy Model Validation', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
#==============================================================================
# PHASE SPACE COMPARISON
#==============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Position space
ax1.plot(X_sindy[:, 0], X_sindy[:, 1], 'k-', linewidth=1, label='True')
ax1.plot(X_sim[:, 0], X_sim[:, 1], 'r--', linewidth=1, label='SINDy')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Orbit (Position Space)')
ax1.set_aspect('equal')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Velocity space
ax2.plot(X_sindy[:, 2], X_sindy[:, 3], 'k-', linewidth=1, label='True')
ax2.plot(X_sim[:, 2], X_sim[:, 3], 'r--', linewidth=1, label='SINDy')
ax2.set_xlabel('vx')
ax2.set_ylabel('vy')
ax2.set_title('Velocity Space')
ax2.set_aspect('equal')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# Part 5: Summary & Discussion

In [ ]:
print("="*70)
print("SUMMARY: LEARNING FROM N-BODY SIMULATIONS")
print("="*70)
print("\n1. PySR (Symbolic Regression)")
print("   - Input: (distance, acceleration) pairs")
print(f"   - Discovered: a = {model_pysr.sympy()}")
print(f"   - Expected:   a = GM/r² = {G*M_sun}/r²")
print("   - Approach: Evolutionary search for symbolic expressions")
print("\n2. SINDy (Sparse Identification of Nonlinear Dynamics)")
print("   - Input: State trajectory [x, y, vx, vy]")
print("   - Discovered equations:")
model_sindy.print()
print("   - Approach: Sparse regression on library of candidate functions")
print("\n" + "-"*70)
print("KEY INSIGHTS:")
print("-"*70)
print("• PySR excels at discovering algebraic relationships (e.g., force laws)")
print("• SINDy excels at discovering dynamical systems (dx/dt = f(x))")
print("• Both struggle with 3-body due to chaotic dynamics and high dimensionality")
print("• Better results require: more data, noise handling, domain knowledge")
print("="*70)

## Exercises

1. **Change orbit parameters**: Try different eccentricities and initial conditions
2. **Add noise**: Add Gaussian noise to measurements and observe effect on learning
3. **Try 3-body with PySR**: Can you discover the force law from 3-body data?
4. **Custom SINDy library**: Add custom functions to capture 1/r² behavior
5. **Compare configurations**: How does the 'figure8' configuration differ from 'triangle'?

In [ ]:
# Your experiments here
